In [1]:
import os
import json
import numpy as np
import pandas as pd
import zarr
from scipy.ndimage import label, uniform_filter
from scipy.optimize import linear_sum_assignment
from skimage.filters import threshold_otsu

DATASET_PATH = '/kaggle/input/competitions/biohub-cell-tracking-during-development'
TEST_DIR = f'{DATASET_PATH}/test'

SCALE = np.array([1.625, 0.40625, 0.40625])  # Z, Y, X um/voxel
DOWNSAMPLE = 4
MAX_LINK_DISTANCE = 15.0  # um
MIN_COMPONENT_VOXELS = 3
DIVISION_DAUGHTER_MAX_DIST = 15.0
MIN_DAUGHTER_SEPARATION = 5.0  # um -- prevents near-duplicate detections being called divisions

def detect_centroids(vol, downsample, min_voxels):
    ds = vol[::downsample, ::downsample, ::downsample]
    smoothed = uniform_filter(ds.astype(np.float32), size=3)
    try:
        threshold = threshold_otsu(smoothed)
    except ValueError:
        threshold = np.percentile(smoothed, 90)
    binary = smoothed > threshold
    labeled, n_features = label(binary)
    centroids = {}
    sizes = {}
    for comp_id in range(1, n_features + 1):
        coords = np.argwhere(labeled == comp_id)
        if coords.shape[0] < min_voxels:
            continue
        centroid = coords.mean(axis=0) * downsample
        centroids[comp_id] = (int(round(centroid[0])), int(round(centroid[1])), int(round(centroid[2])))
        sizes[comp_id] = coords.shape[0]
    return centroids, sizes

def link_frames(prev_centroids, prev_sizes, curr_centroids, curr_sizes, scale, max_dist):
    prev_ids = list(prev_centroids.keys())
    curr_ids = list(curr_centroids.keys())
    if not prev_ids or not curr_ids:
        return [], None, prev_ids, curr_ids
    prev_coords = np.array([prev_centroids[pid] for pid in prev_ids], dtype=np.float64)
    curr_coords = np.array([curr_centroids[cid] for cid in curr_ids], dtype=np.float64)
    prev_phys = prev_coords * scale
    curr_phys = curr_coords * scale
    dist = np.sqrt(((prev_phys[:, None] - curr_phys[None, :]) ** 2).sum(axis=2))
    prev_sz = np.array([prev_sizes[pid] for pid in prev_ids], dtype=np.float64)
    curr_sz = np.array([curr_sizes[cid] for cid in curr_ids], dtype=np.float64)
    size_diff = np.abs(prev_sz[:, None] - curr_sz[None, :])
    size_penalty = size_diff / (prev_sz[:, None] + curr_sz[None, :] + 1e-6)
    cost = dist + 5.0 * size_penalty
    row_ind, col_ind = linear_sum_assignment(cost)
    matches = []
    for ri, ci in zip(row_ind, col_ind):
        if dist[ri, ci] <= max_dist:
            matches.append((prev_ids[ri], curr_ids[ci]))
    return matches, dist, prev_ids, curr_ids

def detect_divisions(dist, prev_ids, curr_ids, matches, curr_centroids, scale,
                      max_daughter_dist=15.0, min_daughter_separation=5.0):
    if dist is None:
        return []
    matched_prev = {p for p, c in matches}
    matched_curr = {c for p, c in matches}
    curr_idx = {cid: i for i, cid in enumerate(curr_ids)}
    unmatched_curr = [cid for cid in curr_ids if cid not in matched_curr]

    candidates_by_parent = {}
    for cid in unmatched_curr:
        ci = curr_idx[cid]
        distances_to_prev = dist[:, ci]
        nearest_prev_i = np.argmin(distances_to_prev)
        nearest_prev_dist = distances_to_prev[nearest_prev_i]
        nearest_prev_id = prev_ids[nearest_prev_i]
        if nearest_prev_dist > max_daughter_dist:
            continue
        if nearest_prev_id not in matched_prev:
            continue
        candidates_by_parent.setdefault(nearest_prev_id, []).append(cid)

    extra_edges = []
    for parent_id, candidate_ids in candidates_by_parent.items():
        if len(candidate_ids) == 1:
            extra_edges.append((parent_id, candidate_ids[0]))
            continue
        coords = np.array([curr_centroids[cid] for cid in candidate_ids], dtype=np.float64) * scale
        pairwise = np.sqrt(((coords[:, None] - coords[None, :]) ** 2).sum(axis=2))
        for i, cid in enumerate(candidate_ids):
            other_dists = np.delete(pairwise[i], i)
            if len(other_dists) == 0 or other_dists.min() >= min_daughter_separation:
                extra_edges.append((parent_id, cid))
    return extra_edges

test_folder_names = sorted(d.replace('.zarr', '') for d in os.listdir(TEST_DIR) if d.endswith('.zarr'))

all_rows = []
for folder_name in test_folder_names:
    zarr_path = os.path.join(TEST_DIR, folder_name + '.zarr')
    with open(os.path.join(zarr_path, '0', 'zarr.json')) as f:
        arr_meta = json.load(f)
    shape = tuple(arr_meta['shape'])
    dtype = np.dtype(arr_meta['data_type'])
    n_t = shape[0]

    prev_centroids = {}
    prev_sizes = {}
    node_id_counter = 1
    comp_id_to_node_id = {}

    root = zarr.open(zarr_path, mode='r')
    image = root['0']

    for t in range(n_t):
        vol = np.asarray(image[t])
        centroids, sizes = detect_centroids(vol, DOWNSAMPLE, MIN_COMPONENT_VOXELS)

        curr_comp_id_to_node_id = {}
        for comp_id, (z, y, x) in centroids.items():
            nid = node_id_counter
            node_id_counter += 1
            curr_comp_id_to_node_id[comp_id] = nid
            all_rows.append({
                'dataset': folder_name, 'row_type': 'node', 'node_id': nid,
                't': t, 'z': z, 'y': y, 'x': x, 'source_id': -1, 'target_id': -1,
            })

        if prev_centroids:
            matches, dist, prev_ids, curr_ids = link_frames(
                prev_centroids, prev_sizes, centroids, sizes, SCALE, MAX_LINK_DISTANCE
            )
            division_edges = detect_divisions(
                dist, prev_ids, curr_ids, matches, centroids, SCALE,
                max_daughter_dist=DIVISION_DAUGHTER_MAX_DIST,
                min_daughter_separation=MIN_DAUGHTER_SEPARATION,
            )
            for prev_comp_id, curr_comp_id in matches + division_edges:
                all_rows.append({
                    'dataset': folder_name, 'row_type': 'edge', 'node_id': -1,
                    't': -1, 'z': -1, 'y': -1, 'x': -1,
                    'source_id': comp_id_to_node_id[prev_comp_id],
                    'target_id': curr_comp_id_to_node_id[curr_comp_id],
                })

        prev_centroids = centroids
        prev_sizes = sizes
        comp_id_to_node_id = curr_comp_id_to_node_id

    print(f'{folder_name}: {node_id_counter - 1} nodes')

submission = pd.DataFrame(all_rows)
submission.index.name = 'id'
submission.to_csv('submission.csv')
print(f'Done. {len(submission)} rows written to submission.csv')

44b6_0113de3b: 550 nodes
44b6_0b24845f: 415 nodes
6bba_05b6850b: 384 nodes
6bba_05db0fb1: 423 nodes
Done. 3087 rows written to submission.csv
